In [49]:
import scipy as sp
import numpy as np
import numpy.linalg as la
import sympy as smp
import scipy.stats as stats
import matplotlib.pyplot as plt

import tensorly as tl

In [50]:
# tensor -> matrix

def unfold(M, mode=0):
    m = np.moveaxis(M, mode, 0).reshape(M.shape[mode], -1)
    return m

mat = np.array([
    [
        [1, 5],
        [3, 7]
    ],
    [
        [2, 6],
        [4, 8]
    ]
])

mat, unfold(mat, 2)


(array([[[1, 5],
         [3, 7]],
 
        [[2, 6],
         [4, 8]]]),
 array([[1, 3, 2, 4],
        [5, 7, 6, 8]]))

In [51]:
def factor_matrices(M):
    factors = []

    for mode in range(M.ndim):
        A = unfold(M, mode)
        U, _, _ = la.svd(A)
        factors.append(U)
    
    return factors

m_fac = factor_matrices(mat)
print(f"{[m.shape for m in m_fac]}")
m_fac

[(2, 2), (2, 2), (2, 2)]


[array([[-0.64142303, -0.7671874 ],
        [-0.7671874 ,  0.64142303]]),
 array([[-0.56672424, -0.82390754],
        [-0.82390754,  0.56672424]]),
 array([[-0.37616823, -0.92655138],
        [-0.92655138,  0.37616823]])]

In [52]:

def refold_0(tensor):
    return np.array([
        [[tensor[i, j + 2 * k] for j in range(2)] for k in range(2)] for i in range(2)
    ])

def refold(tensor, mode, shape):
    full_shape = [shape[mode]] + [shape[i] for i in range(len(shape)) if i != mode]
    return tensor.reshape(full_shape).swapaxes(0, mode)

def mode_n_product(T, U, mode):
    A = unfold(T, mode)
    result = U @ A
    
    # refold mat
    new_shape = list(T.shape)
    new_shape[mode] = U.shape[1]
    return refold(result, mode, new_shape)


def core_tensor(M, factors):
    G = M.copy()
    for mode, U in enumerate(factors):
        G = mode_n_product(G, U, mode)
    return G

G = core_tensor(mat, m_fac)
G

array([[[-1.42253953e+01,  4.61793060e-03],
        [ 1.60125603e-02,  5.43770692e-01]],

       [[ 8.28025332e-03,  1.11585148e+00],
        [ 2.38589095e-01,  2.00114739e-01]]])

In [53]:
def hosvd(M):
    factors = factor_matrices(M)
    G = core_tensor(M, factors)
    return G#, factors


In [54]:
tensor = np.array([
    [
        [0, 1],
        [1, 0]
    ],
    [
        [1, 1],
        [0, 0]
    ]
])

ct_0 = hosvd(tensor)
ct_1 = tl.decomposition.tucker(tensor, tensor.ndim).core
print(f"HOSVD:\n{ct_0}")
print()
print(f"Tucker Decomp (Tensorly):\n{ct_1}")

np.array_equal(ct_0, ct_1)

HOSVD:
[[[ 1.5 -0.5]
  [-0.5 -0.5]]

 [[ 0.5  0.5]
  [ 0.5  0.5]]]

Tucker Decomp (Tensorly):
[[[ 1.5  0.5]
  [ 0.5 -0.5]]

 [[-0.5  0.5]
  [ 0.5 -0.5]]]


False

In [55]:
A_1234 = np.array([
    [
        [1, 2],
        [3, 4]
    ],
    [
        [5, 6],
        [7, 8]
    ]
])

A = np.array([
    [1, 2],
    [3, 4]
])

# temp = mode_n_product(ct, A, 0)
# hosvd(temp), ct
# temp, A, ct


t_f = unfold(A_1234)
t_rf = refold(t_f, mode=0, shape=(2, 2, 2))

t_f, t_rf

(array([[1, 2, 3, 4],
        [5, 6, 7, 8]]),
 array([[[1, 2],
         [3, 4]],
 
        [[5, 6],
         [7, 8]]]))

In [56]:
unfold = tl.unfold(A_1234, mode=0)
fold = tl.fold(unfold, mode=0, shape=(2, 2, 2))
unfold, fold

(array([[1, 2, 3, 4],
        [5, 6, 7, 8]]),
 array([[[1, 2],
         [3, 4]],
 
        [[5, 6],
         [7, 8]]]))

## Core Tensors: Modal Products of $W$ with $SL(2, \mathbb{C})$

In [ ]:
import scipy.linalg as sla

e = np.array([[0, 1], [0, 0]])
f = np.array([[0, 0], [1, 0]])
h = np.array([[1, 0], [0, -1]])

def form_sl2(α, β, γ):
    return sla.expm(α * e + β * f + γ * h)

# W state as 2x2x2 tensor
W = np.array([0, np.sqrt(3), np.sqrt(3), 0, np.sqrt(3), 0, 0, 0]).reshape((2, 2, 2))
W

### 1. $W \times_0 \exp(e)$

In [ ]:
E = form_sl2(1, 0, 0)

mp_1 = tl.tenalg.multi_mode_dot(W, [E], modes=[0])
ct_1 = tl.decomposition.tucker(mp_1, 2).core
ct_1

### 2. $W \times_1 \exp(e)$

In [ ]:
mp_2 = tl.tenalg.multi_mode_dot(W, [E], modes=[1])
ct_2 = tl.decomposition.tucker(mp_2, 2).core
ct_2

### 3. $W \times_2 \exp(e)$

In [ ]:
mp_3 = tl.tenalg.multi_mode_dot(W, [E], modes=[2])
ct_3 = tl.decomposition.tucker(mp_3, 2).core
ct_3

### 4. $W \times_0 \exp(f)$

In [ ]:
F = form_sl2(0, 1, 0)

mp_4 = tl.tenalg.multi_mode_dot(W, [F], modes=[0])
ct_4 = tl.decomposition.tucker(mp_4, 2).core
ct_4

### 5. $W \times_0 \exp(h)$

In [ ]:
H = form_sl2(0, 0, 1)

mp_5 = tl.tenalg.multi_mode_dot(W, [H], modes=[0])
ct_5 = tl.decomposition.tucker(mp_5, 2).core
ct_5

### 6. $W \times_0 \exp(e) \times_1 \exp(e) \times_2 \exp(e)$

In [ ]:
mp_6 = tl.tenalg.multi_mode_dot(W, [E, E, E], modes=[0, 1, 2])
ct_6 = tl.decomposition.tucker(mp_6, 2).core
ct_6

### 7. $W \times_0 \exp(f) \times_1 \exp(f) \times_2 \exp(f)$

In [ ]:
mp_7 = tl.tenalg.multi_mode_dot(W, [F, F, F], modes=[0, 1, 2])
ct_7 = tl.decomposition.tucker(mp_7, 2).core
ct_7

### 8. $W \times_0 \exp(h) \times_1 \exp(h) \times_2 \exp(h)$

In [ ]:
mp_8 = tl.tenalg.multi_mode_dot(W, [H, H, H], modes=[0, 1, 2])
ct_8 = tl.decomposition.tucker(mp_8, 2).core
ct_8

### 9. $W \times_0 \exp(e) \times_1 \exp(f) \times_2 \exp(h)$

In [ ]:
mp_9 = tl.tenalg.multi_mode_dot(W, [E, F, H], modes=[0, 1, 2])
ct_9 = tl.decomposition.tucker(mp_9, 2).core
ct_9

### 10. $W \times_0 \exp(e+f) \times_1 \exp(e+f) \times_2 \exp(e+f)$

In [ ]:
EF = form_sl2(1, 1, 0)

mp_10 = tl.tenalg.multi_mode_dot(W, [EF, EF, EF], modes=[0, 1, 2])
ct_10 = tl.decomposition.tucker(mp_10, 2).core
ct_10

### Summary

In [ ]:
cores = [ct_1, ct_2, ct_3, ct_4, ct_5, ct_6, ct_7, ct_8, ct_9, ct_10]
labels = [
    "W ×₀ exp(e)", "W ×₁ exp(e)", "W ×₂ exp(e)",
    "W ×₀ exp(f)", "W ×₀ exp(h)",
    "W ×₀₁₂ exp(e)", "W ×₀₁₂ exp(f)", "W ×₀₁₂ exp(h)",
    "W ×₀e ×₁f ×₂h", "W ×₀₁₂ exp(e+f)"
]

# compare core tensors of W across modes (permutation symmetry check)
print("=== Single-mode exp(e): permutation symmetry ===")
print(f"mode 0 ≈ mode 1: {np.allclose(np.sort(ct_1.flat), np.sort(ct_2.flat))}")
print(f"mode 0 ≈ mode 2: {np.allclose(np.sort(ct_1.flat), np.sort(ct_3.flat))}")
print()

# compare Frobenius norms
print("=== Frobenius norms ===")
for label, ct in zip(labels, cores):
    print(f"{label:25s}  ||G|| = {la.norm(ct):.6f}")
print()

# compare to W core tensor baseline
ct_W = tl.decomposition.tucker(W, 2).core
print(f"{'W (baseline)':25s}  ||G|| = {la.norm(ct_W):.6f}")
print()
print("W core tensor:")
ct_W